# EDA: All Methods Results

Phan tich ket qua `all_methods_results.csv` cho tat ca metric:

- `acc`: accuracy, cao hon tot hon
- `f1`: macro F1, cao hon tot hon
- `auc`: ROC-AUC, cao hon tot hon
- `ap`: average precision, cao hon tot hon
- `eer`: equal error rate, thap hon tot hon

Notebook nay khong chi nhin F1; moi ranking/delta/heatmap deu duoc tinh cho tat ca metric.

In [52]:
# !pip install -q plotly

In [64]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

px.defaults.template = "plotly_white"
px.defaults.width = 1150
px.defaults.height = 600

## Load Data

In [65]:
CSV_PATH = Path("results_data/all_methods_results.csv")

df = pd.read_csv(CSV_PATH)
metric_cols = ["acc", "f1", "auc", "ap", "eer"]
higher_is_better = {"acc": True, "f1": True, "auc": True, "ap": True, "eer": False}

df["level_raw"] = df["level"].astype(str)
df["level_num"] = pd.to_numeric(df["level"], errors="coerce")
df["is_train"] = df["level_raw"].str.lower().eq("train") | df["corruption"].eq("train_ffpp")
df_eval = df[~df["is_train"]].copy()

print("full shape:", df.shape)
print("eval shape:", df_eval.shape)
print("methods:", sorted(df_eval["method"].unique()))
print("corruptions:", sorted(df_eval["corruption"].unique()))
print("levels:", sorted(df_eval["level_num"].dropna().astype(int).unique()))
display(df.head())

full shape: (261, 12)
eval shape: (260, 12)
methods: ['bca', 'boost_adapter', 'compact_cache_adapter', 'crg', 'dmn', 'dota', 'dpe', 'dynaprompt', 'freetta', 'linear_probe', 'online_confident_cache_adapter', 'prototype_linear_tta', 'tip_adapter']
corruptions: ['color_contrast', 'color_saturation', 'gaussian_blur', 'resize']
levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


,level,corruption,method,acc,f1,auc,ap,eer,eer_threshold,level_raw,level_num,is_train
0,train,train_ffpp,linear_probe,0.934708,0.733880,0.948573,0.995174,0.124862,0.865461,train,NaN,True
1,1,color_contrast,linear_probe,0.805440,0.576033,0.698977,0.940008,0.358568,0.873829,1,1.0,False
2,1,color_contrast,tip_adapter,0.859913,0.498405,0.687320,0.933886,0.363698,0.890244,1,1.0,False
3,1,color_contrast,boost_adapter,0.864011,0.481550,0.722861,0.946078,0.335973,0.868635,1,1.0,False
4,1,color_contrast,compact_cache_adapter,0.561182,0.478462,0.624617,0.911982,0.404082,0.491082,1,1.0,False


## Data Quality Check

In [66]:
display(df.isna().sum().to_frame("missing"))

if "error" in df.columns:
    error_rows = df[df["error"].notna()]
    print("error rows:", len(error_rows))
    display(error_rows[["level", "corruption", "method", "error"]].head(50))
else:
    print("No error column found.")

,missing
level,0
corruption,0
method,0
acc,0
f1,0
auc,0
ap,0
eer,0
eer_threshold,0
level_raw,0


No error column found.


## Overall Ranking Across All Metrics

Cach doc:
- Moi metric duoc rank rieng.
- `mean_rank` la trung binh rank cua `acc`, `f1`, `auc`, `ap`, `eer`.
- `eer` duoc rank nguoc lai vi thap hon la tot hon.

Nhan xet hien tai tu CSV nay:
- `dmn` dung dau tong hop vi rat manh o `auc`, `ap`, va `eer`, du F1 khong cao.
- `prototype_linear_tta` tot nhat ve `f1`, gan nhu ngang baseline ve cac metric con lai.
- `boost_adapter` tot nhat ve `acc`, va rat manh o `auc/ap/eer`, nhung F1 giam nhieu.
- `linear_probe` la baseline rat kho vuot neu uu tien F1.
- `dota` va `dynaprompt` dang kem on dinh nhat, khong nen dua vao main table neu khong co ablation rieng.

In [67]:
summary = df_eval.groupby("method", as_index=False)[metric_cols].mean()

for metric in metric_cols:
    summary[f"rank_{metric}"] = summary[metric].rank(ascending=not higher_is_better[metric], method="min")

rank_cols = [f"rank_{m}" for m in metric_cols]
summary["mean_rank"] = summary[rank_cols].mean(axis=1)
summary = summary.sort_values("mean_rank")
method_order = summary["method"].tolist()

summary_view = summary[["method"] + metric_cols + rank_cols + ["mean_rank"]]
display(summary_view.style.format({**{m: "{:.4f}" for m in metric_cols}, **{r: "{:.2f}" for r in rank_cols + ["mean_rank"]}}))

summary_long = summary.melt(id_vars="method", value_vars=metric_cols, var_name="metric", value_name="value")
fig = px.bar(
    summary_long,
    x="method",
    y="value",
    color="method",
    facet_col="metric",
    facet_col_wrap=1,
    category_orders={"method": method_order, "metric": metric_cols},
    title="Average Metrics By Method",
    text_auto=".3f",
)
fig.update_layout(showlegend=False, xaxis_tickangle=-150, width=1050, height=2160)
fig.show()

,method,acc,f1,auc,ap,eer,rank_acc,rank_f1,rank_auc,rank_ap,rank_eer,mean_rank
4,dmn,0.8617,0.4877,0.6565,0.9218,0.3845,4.00,8.00,1.00,1.00,1.00,3.00
11,prototype_linear_tta,0.8413,0.5271,0.6342,0.9152,0.4051,7.00,1.00,3.00,3.00,3.00,3.40
1,boost_adapter,0.8672,0.4712,0.6560,0.9211,0.3855,1.00,11.00,2.00,2.00,2.00,3.60
9,linear_probe,0.8412,0.5268,0.6337,0.9150,0.4054,8.00,2.00,4.00,4.00,4.00,4.40
10,online_confident_cache_adapter,0.8418,0.5265,0.6219,0.9115,0.4150,6.00,3.00,6.00,6.00,6.00,5.40
8,freetta,0.8173,0.5135,0.6252,0.9130,0.4086,11.00,5.00,5.00,5.00,5.00,6.20
12,tip_adapter,0.8665,0.4736,0.6101,0.9010,0.4171,2.00,10.00,7.00,8.00,7.00,6.80
3,crg,0.8346,0.5037,0.5901,0.9084,0.4570,9.00,6.00,8.00,7.00,8.00,7.60
0,bca,0.8621,0.4780,0.5037,0.8629,0.5067,3.00,9.00,10.00,11.00,10.00,8.60
2,compact_cache_adapter,0.6710,0.5023,0.5466,0.8816,0.4733,12.00,7.00,9.00,9.00,9.00,9.20


## Best Methods Per Metric

Day la bang de tranh ket luan dua tren mot metric duy nhat.

Nhan xet:
- Neu uu tien `acc`: `boost_adapter` va `tip_adapter` thuong dung dau, nhung co the hy sinh F1.
- Neu uu tien `f1`: `prototype_linear_tta`, `linear_probe`, va `online_confident_cache_adapter` gan nhau nhat.
- Neu uu tien ranking/threshold-free metric nhu `auc`, `ap`, `eer`: `dmn` va `boost_adapter` dang manh hon baseline.

In [57]:
for metric in metric_cols:
    ranking = summary[["method", metric]].sort_values(metric, ascending=not higher_is_better[metric]).reset_index(drop=True)
    ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
    direction = "higher is better" if higher_is_better[metric] else "lower is better"
    print(f"\nRanking by {metric.upper()} ({direction})")
    display(ranking.style.format({metric: "{:.4f}"}))


Ranking by ACC (higher is better)


,rank,method,acc
0,1,boost_adapter,0.8672
1,2,tip_adapter,0.8665
2,3,bca,0.8621
3,4,dmn,0.8617
4,5,dynaprompt,0.8554
5,6,online_confident_cache_adapter,0.8418
6,7,prototype_linear_tta,0.8413
7,8,linear_probe,0.8412
8,9,crg,0.8346
9,10,dpe,0.8187



Ranking by F1 (higher is better)


,rank,method,f1
0,1,prototype_linear_tta,0.5271
1,2,linear_probe,0.5268
2,3,online_confident_cache_adapter,0.5265
3,4,dpe,0.5182
4,5,freetta,0.5135
5,6,crg,0.5037
6,7,compact_cache_adapter,0.5023
7,8,dmn,0.4877
8,9,bca,0.4780
9,10,tip_adapter,0.4736



Ranking by AUC (higher is better)


,rank,method,auc
0,1,dmn,0.6565
1,2,boost_adapter,0.6560
2,3,prototype_linear_tta,0.6342
3,4,linear_probe,0.6337
4,5,freetta,0.6252
5,6,online_confident_cache_adapter,0.6219
6,7,tip_adapter,0.6101
7,8,crg,0.5901
8,9,compact_cache_adapter,0.5466
9,10,bca,0.5037



Ranking by AP (higher is better)


,rank,method,ap
0,1,dmn,0.9218
1,2,boost_adapter,0.9211
2,3,prototype_linear_tta,0.9152
3,4,linear_probe,0.9150
4,5,freetta,0.9130
5,6,online_confident_cache_adapter,0.9115
6,7,crg,0.9084
7,8,tip_adapter,0.9010
8,9,compact_cache_adapter,0.8816
9,10,dpe,0.8684



Ranking by EER (lower is better)


,rank,method,eer
0,1,dmn,0.3845
1,2,boost_adapter,0.3855
2,3,prototype_linear_tta,0.4051
3,4,linear_probe,0.4054
4,5,freetta,0.4086
5,6,online_confident_cache_adapter,0.4150
6,7,tip_adapter,0.4171
7,8,crg,0.4570
8,9,compact_cache_adapter,0.4733
9,10,bca,0.5067


## Metric Trends Across Corruption Levels

Chart nay cho biet method nao on dinh khi corruption level tang.

Dieu can de y:
- Neu `acc` cao nhung `f1` thap, method co the dang bias ve majority class.
- Neu `auc/ap/eer` tot nhung `f1` kem, ranking score tot nhung threshold/calibration chua phu hop.
- Neu metric giam nhanh theo level, method nhay voi corruption severity.

In [58]:
trend_long = (
    df_eval.groupby(["level_num", "method"], as_index=False)[metric_cols]
    .mean()
    .melt(id_vars=["level_num", "method"], value_vars=metric_cols, var_name="metric", value_name="value")
)

fig = px.line(
    trend_long,
    x="level_num",
    y="value",
    color="method",
    facet_col="metric",
    facet_col_wrap=3,
    markers=True,
    category_orders={"method": method_order, "metric": metric_cols},
    title="Metric Trends Across Corruption Levels",
)
fig.update_layout(xaxis_title="Corruption level", yaxis_title="Metric value")
fig.show()

## Corruption Breakdown For All Metrics

Nhan xet hien tai:
- `boost_adapter` rat manh tren `gaussian_blur` va `resize` neu nhin `acc/auc/ap/eer`.
- `prototype_linear_tta` tot hon cho F1 tren `color_contrast` va `color_saturation`.
- `online_confident_cache_adapter` co F1 tot nhat tren blur/resize, nhung AUC/EER khong bang `boost_adapter`.

In [59]:
corruption_long = (
    df_eval.groupby(["corruption", "method"], as_index=False)[metric_cols]
    .mean()
    .melt(id_vars=["corruption", "method"], value_vars=metric_cols, var_name="metric", value_name="value")
)

fig = px.bar(
    corruption_long,
    x="method",
    y="value",
    color="method",
    facet_row="metric",
    facet_col="corruption",
    category_orders={"method": method_order, "metric": metric_cols},
    title="Mean Metrics By Corruption And Method",
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30, height=1300)
fig.show()

## Heatmaps For Every Metric

Moi heatmap la method x condition (`level + corruption`).

- Mau xanh hon la tot hon.
- Rieng `eer` dung colorscale dao nguoc vi thap hon la tot hon.

In [60]:
heat = df_eval.copy()
heat["condition"] = "L" + heat["level_num"].astype(int).astype(str) + " | " + heat["corruption"]

for metric in metric_cols:
    pivot = heat.pivot_table(index="method", columns="condition", values=metric, aggfunc="mean").reindex(method_order)
    fig = px.imshow(
        pivot,
        aspect="auto",
        color_continuous_scale="RdYlGn_r" if metric == "eer" else "RdYlGn",
        title=f"{metric.upper()} Heatmap By Method And Condition",
        text_auto=".3f",
    )
    fig.update_layout(xaxis_title="Condition", yaxis_title="Method")
    fig.show()

## Delta Against Linear Probe

Day la phan quan trong nhat de xem TTA co that su giup khong.

Nhan xet hien tai:
- `prototype_linear_tta` gan nhu chi nhinh hon baseline rat nhe, nhung it pha baseline nhat.
- `dmn` va `boost_adapter` cai thien `acc`, `auc`, `ap`, `eer`, nhung lam giam F1. Dieu nay goi y van de threshold/class imbalance.
- `tip_adapter` tang accuracy nhung giam F1, nen khong nen chi report accuracy.
- `dota` lam te hon hau het metric.

In [61]:
key_cols = ["level_num", "corruption"]
baseline = (
    df_eval[df_eval["method"].eq("linear_probe")][key_cols + metric_cols]
    .rename(columns={m: f"baseline_{m}" for m in metric_cols})
)

delta = df_eval.merge(baseline, on=key_cols, how="left")
for metric in metric_cols:
    delta[f"delta_{metric}"] = delta[metric] - delta[f"baseline_{metric}"]

delta_cols = [f"delta_{m}" for m in metric_cols]
delta_summary = delta[~delta["method"].eq("linear_probe")].groupby("method", as_index=False)[delta_cols].mean()

for metric in metric_cols:
    col = f"delta_{metric}"
    delta_summary[f"rank_{metric}"] = delta_summary[col].rank(ascending=not higher_is_better[metric])

delta_summary["mean_delta_rank"] = delta_summary[[f"rank_{m}" for m in metric_cols]].mean(axis=1)
delta_summary = delta_summary.sort_values("mean_delta_rank")

display(delta_summary.style.format({**{c: "{:+.4f}" for c in delta_cols}, "mean_delta_rank": "{:.2f}"}))

delta_long = delta_summary.melt(id_vars="method", value_vars=delta_cols, var_name="metric", value_name="delta")
delta_long["metric"] = delta_long["metric"].str.replace("delta_", "", regex=False)

fig = px.bar(
    delta_long,
    x="method",
    y="delta",
    color="delta",
    facet_col="metric",
    facet_col_wrap=3,
    color_continuous_scale="RdYlGn",
    category_orders={"method": delta_summary["method"].tolist(), "metric": metric_cols},
    title="Mean Metric Delta vs Linear Probe",
    text_auto="+.3f",
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(xaxis_tickangle=-30)
fig.show()

,method,delta_acc,delta_f1,delta_auc,delta_ap,delta_eer,rank_acc,rank_f1,rank_auc,rank_ap,rank_eer,mean_delta_rank
4,dmn,+0.0204,-0.0391,+0.0228,+0.0068,-0.0209,4.000000,7.000000,1.000000,1.000000,1.000000,2.80
1,boost_adapter,+0.0260,-0.0556,+0.0223,+0.0061,-0.0199,1.000000,10.000000,2.000000,2.000000,2.000000,3.40
10,prototype_linear_tta,+0.0000,+0.0003,+0.0005,+0.0001,-0.0004,7.000000,1.000000,3.000000,3.000000,3.000000,3.40
9,online_confident_cache_adapter,+0.0005,-0.0003,-0.0118,-0.0035,+0.0095,6.000000,2.000000,5.000000,5.000000,5.000000,4.60
8,freetta,-0.0240,-0.0133,-0.0085,-0.0020,+0.0032,10.000000,4.000000,4.000000,4.000000,4.000000,5.20
11,tip_adapter,+0.0252,-0.0532,-0.0235,-0.0140,+0.0116,2.000000,9.000000,6.000000,7.000000,6.000000,6.00
3,crg,-0.0067,-0.0231,-0.0435,-0.0066,+0.0515,8.000000,5.000000,7.000000,6.000000,7.000000,6.60
0,bca,+0.0209,-0.0488,-0.1300,-0.0521,+0.1012,3.000000,8.000000,9.000000,10.000000,9.000000,7.80
2,compact_cache_adapter,-0.1702,-0.0244,-0.0871,-0.0334,+0.0679,11.000000,6.000000,8.000000,8.000000,8.000000,8.20
6,dpe,-0.0225,-0.0086,-0.1382,-0.0466,+0.1097,9.000000,3.000000,10.000000,9.000000,10.000000,8.20


## Best Method Per Condition For Each Metric

Bang nay tra loi cau hoi: voi tung `level + corruption`, method nao thang theo moi metric.

Dung bang nay de viet nhan xet theo tung corruption, thay vi chi lay mean tong.

In [62]:
best_rows = []
for metric in metric_cols:
    best_metric = (
        df_eval.sort_values(metric, ascending=not higher_is_better[metric])
        .groupby(["level_num", "corruption"], as_index=False)
        .first()[["level_num", "corruption", "method", metric]]
        .rename(columns={metric: "value"})
    )
    best_metric["metric"] = metric
    best_rows.append(best_metric)

best_all = pd.concat(best_rows, ignore_index=True)
display(best_all.sort_values(["metric", "level_num", "corruption"]).style.format({"value": "{:.4f}"}))

win_counts = best_all.value_counts(["metric", "method"]).rename("wins").reset_index()
fig = px.bar(
    win_counts,
    x="method",
    y="wins",
    color="method",
    facet_col="metric",
    facet_col_wrap=3,
    category_orders={"method": method_order, "metric": metric_cols},
    title="Best Method Count Per Metric",
    text_auto=True,
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

,level_num,corruption,method,value,metric
0,1.000000,color_contrast,boost_adapter,0.8640,acc
1,1.000000,color_saturation,boost_adapter,0.8637,acc
2,1.000000,gaussian_blur,boost_adapter,0.8669,acc
3,1.000000,resize,boost_adapter,0.8670,acc
4,2.000000,color_contrast,boost_adapter,0.8658,acc
5,2.000000,color_saturation,boost_adapter,0.8664,acc
6,2.000000,gaussian_blur,boost_adapter,0.8678,acc
7,2.000000,resize,boost_adapter,0.8678,acc
8,3.000000,color_contrast,boost_adapter,0.8670,acc
9,3.000000,color_saturation,boost_adapter,0.8681,acc


## Final Takeaways

Ket luan tu file ket qua hien tai:

1. Neu report theo ranking score (`auc`, `ap`, `eer`), `dmn` va `boost_adapter` la hai ung vien manh nhat.
2. Neu report theo macro-F1, `prototype_linear_tta` chi nhinh hon `linear_probe` rat nhe; TTA chua tao gain lon ve F1.
3. `boost_adapter`, `tip_adapter`, `bca`, `dynaprompt` co accuracy cao nhung F1 thap hon baseline, kha nang do class imbalance va threshold/calibration.
4. `dota` khong phu hop voi setup hien tai, co the bo khoi main result va dua vao ablation.
5. Nen report nhieu metric cung luc. Neu chi report accuracy, cache-based methods nhin rat tot; neu report F1, baseline/prototype-linear lai canh tranh hon.

In [63]:
OUT_DIR = Path("summary")
OUT_DIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(OUT_DIR / "method_metric_summary.csv", index=False)
delta_summary.to_csv(OUT_DIR / "delta_vs_linear_probe.csv", index=False)
best_all.to_csv(OUT_DIR / "best_method_per_condition.csv", index=False)

print("saved summaries to", OUT_DIR.resolve())

saved summaries to /Users/hoavien/Documents/tta/code/deepfake_tta/eda/summary
